# 03 — Modelo de referencia: XGBoost + skforecast

**RESPIR-AI · TFG** — Predicción horaria de la tasa de consumo de oxígeno (OUR) del reactor biológico 1.

Este cuaderno entrena el modelo de referencia clásico: un `XGBRegressor` envuelto en
`ForecasterRecursiveMultiSeries` de *skforecast* (predicción recursiva multi-paso con 48 retardos).
Se evalúan tres escenarios sobre el protocolo de evaluación v2 (36 orígenes rolling-origin en el
conjunto de test, horizontes H ∈ {6, 12, 24, 48}):

- **E1 — Univariante**: solo los 48 retardos de la propia OUR.
- **E2 — Meteo naïve**: covariables meteorológicas cuyo valor futuro se aproxima por persistencia
  (último valor observado), simulando la ausencia de pronóstico.
- **E3 — Meteo real («oráculo»)**: covariables meteorológicas futuras reales, equivalentes a un
  pronóstico perfecto. Es una cota superior del beneficio alcanzable con exógenas meteorológicas.

Los hiperparámetros se seleccionan mediante validación *walk-forward* sobre el bloque de
validación (27 orígenes, paso 48 h) con una rejilla reducida
(`learning_rate` ∈ {0.05, 0.1}, `max_depth` ∈ {4, 6}, `n_estimators` ∈ {300, 600}).
El modelo final se reentrena con train+val. Semilla fija: 42.

## 1. Infraestructura común de evaluación

Módulo compartido por los tres cuadernos de modelos: carga de datos, orígenes del protocolo, métricas agregadas y por ventana.

In [1]:
# Raíz del repositorio como cwd (permite ejecutar desde notebooks/ o desde la raíz)
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"common_eval.py").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# Módulo compartido de evaluación (protocolo v2) — ver common_eval.py en la raíz del repo
from common_eval import *
import common_eval as CE


## 2. Carga de datos y verificación del protocolo

In [2]:
import warnings, time, itertools, pickle, os
import numpy as np, pandas as pd
from xgboost import XGBRegressor
from skforecast.recursive import ForecasterRecursiveMultiSeries
warnings.filterwarnings("ignore")
np.random.seed(42)

df, proto = load_all()
origins = get_origins(df, proto)          # 36 orígenes del protocolo v2
Y = true_targets(df, origins)             # (36, 48) valores reales de OUR
masks = split_masks(df, proto)
train_end = proto['particion']['train']['fin']
val_end   = proto['particion']['val']['fin']

# Serie objetivo como serie horaria continua con NaN fuera de segmentos útiles:
# skforecast (dropna_from_series=True) descarta las muestras cuyos retardos
# cruzan un hueco, de modo que nunca se mezclan segmentos.
y_full = df["our"].where(df.segment_id >= 0).asfreq('h')
exog_full = df[METEO].astype(float).asfreq('h')
print(f"train hasta {train_end} · val hasta {val_end} · {len(origins)} orígenes de test")

train hasta 2025-05-23 15:00:00+00:00 · val hasta 2025-07-26 17:00:00+00:00 · 36 orígenes de test


## 3. Búsqueda ligera de hiperparámetros (walk-forward en validación)

Se generan orígenes de validación con el mismo criterio del protocolo (contexto mínimo 48 h, 48 h de futuro dentro del segmento, paso 48 h) y se elige la combinación con menor MAE medio a 48 h.

In [3]:
# Orígenes walk-forward en validación
val_mask = masks['val'].to_numpy(); seg = df.segment_id.to_numpy()
val_origins, last = [], -10**9
for i in np.where(val_mask)[0]:
    if i - last < 48 or seg[i] < 0: continue
    if i-47 < 0 or (seg[i-47:i+1] != seg[i]).any(): continue
    if i+48 >= len(df) or (seg[i+1:i+49] != seg[i]).any(): continue
    val_origins.append(i); last = i
Y_val = np.stack([df["our"].iloc[i+1:i+49].to_numpy() for i in val_origins])

def make_forecaster(params):
    return ForecasterRecursiveMultiSeries(
        estimator=XGBRegressor(**params, random_state=42, n_jobs=8,
                               objective='reg:squarederror'),
        lags=48, encoding=None, dropna_from_series=True)

def predict_windows(f, origin_ilocs, exog_mode=None):
    """exog_mode: None | 'real' | 'naive' (persistencia del último valor observado)."""
    preds = []
    for i in origin_ilocs:
        lw = pd.DataFrame({"our": df["our"].iloc[i-47:i+1]}).asfreq('h')
        ex = None
        if exog_mode == 'real':
            ex = exog_full.iloc[i+1:i+49]
        elif exog_mode == 'naive':
            ex = exog_full.iloc[i+1:i+49].copy()
            ex.loc[:, :] = np.tile(exog_full.iloc[i].to_numpy(), (48, 1))
        p = f.predict(steps=48, last_window=lw, exog=ex, levels="our")
        preds.append(p['pred'].to_numpy())
    return np.stack(preds)

grid = list(itertools.product([0.05, 0.1], [4, 6], [300, 600]))
results_hp = {}
for use_exog, tag in [(False, 'univar'), (True, 'exog')]:
    best = None
    for lr, md, ne in grid:
        params = dict(learning_rate=lr, max_depth=md, n_estimators=ne)
        f = make_forecaster(params)
        f.fit(series=pd.DataFrame({"our": y_full.loc[:train_end]}),
              exog=exog_full.loc[:train_end] if use_exog else None)
        P = predict_windows(f, val_origins, exog_mode='real' if use_exog else None)
        mae = float(np.mean(np.abs(Y_val - P)))
        if best is None or mae < best[0]: best = (mae, params)
    results_hp[tag] = best
    print(tag, '→ mejor MAE val:', round(best[0], 3), best[1])

univar → mejor MAE val: 3.331 {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300}
exog → mejor MAE val: 3.412 {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300}


Resultado de la búsqueda (ejecución real): tanto en univariante como con exógenas la mejor
combinación fue `learning_rate=0.05, max_depth=4, n_estimators=300`
(MAE val 3.331 y 3.412 respectivamente) — el modelo más regularizado de la rejilla,
coherente con una serie ruidosa y un horizonte largo.

## 4. Entrenamiento final (train+val) y evaluación en los 36 orígenes de test

In [4]:
test_ilocs = [o['iloc'] for o in origins]
all_rows, all_win, cost_rows = [], [], []

def run_baseline(escenario, params, use_exog, exog_mode, desc):
    t0 = time.perf_counter()
    f = make_forecaster(params)
    f.fit(series=pd.DataFrame({"our": y_full.loc[:val_end]}),
          exog=exog_full.loc[:val_end] if use_exog else None)
    t_train = time.perf_counter() - t0
    t0 = time.perf_counter()
    P = predict_windows(f, test_ilocs, exog_mode=exog_mode)
    t_inf = (time.perf_counter() - t0) / len(test_ilocs)
    rows, wins = evaluate_preds(P, Y, "xgboost", escenario, t_inf, 48, origins,
                                extra={"covariables": desc, "hp": str(params)})
    all_rows.extend(rows); all_win.extend(wins)
    cost_rows.append({"modelo": "xgboost", "escenario": escenario,
                      "n_parametros": np.nan, "t_train_s": t_train,
                      "t_inf_ventana_cpu_s": t_inf, "t_inf_ventana_gpu_s": np.nan,
                      "mem_gpu_pico_MB": 0.0, "dispositivo": "CPU"})
    return P

P_e1 = run_baseline("E1_univariante", results_hp['univar'][1], False, None, "ninguna (48 lags)")
P_e2 = run_baseline("E2_meteo_naive", results_hp['exog'][1], True, 'naive', "meteo futura por persistencia")
P_e3 = run_baseline("E3_meteo_real",  results_hp['exog'][1], True, 'real',  "meteo futura real (oráculo)")

df_base = pd.DataFrame(all_rows)
os.makedirs("results", exist_ok=True)
df_base.to_csv("results/resultados_baseline.csv", index=False)
print(df_base[['escenario','H','MAE','RMSE','MAPE','R2']].round(3).to_string(index=False))

     escenario  H   MAE  RMSE   MAPE    R2
E1_univariante  6 3.428 4.458 16.981 0.355
E1_univariante 12 3.730 4.952 17.863 0.358
E1_univariante 24 3.921 5.054 18.155 0.246
E1_univariante 48 4.388 5.487 20.334 0.148
E2_meteo_naive  6 3.554 4.564 17.870 0.324
E2_meteo_naive 12 3.777 5.018 18.375 0.341
E2_meteo_naive 24 3.864 5.004 18.218 0.260
E2_meteo_naive 48 4.312 5.432 20.339 0.165
 E3_meteo_real  6 3.491 4.510 17.554 0.340
 E3_meteo_real 12 3.760 4.986 18.264 0.350
 E3_meteo_real 24 3.927 5.021 18.377 0.255
 E3_meteo_real 48 4.372 5.440 20.456 0.162


## 5. Resultados obtenidos (ejecución real)

| Escenario | H=6 | H=12 | H=24 | H=48 |
|---|---|---|---|---|
| E1 univariante (MAE) | 3.428 | 3.730 | 3.921 | 4.388 |
| E2 meteo naïve (MAE) | 3.554 | 3.777 | 3.864 | 4.312 |
| E3 meteo real (MAE) | 3.491 | 3.760 | 3.927 | 4.372 |

**Lectura**: las covariables meteorológicas apenas alteran el error del baseline
(diferencias < 0.08 MAE entre escenarios en la mayoría de horizontes, sin patrón
consistente a favor del «oráculo»). La señal predictiva dominante para XGBoost
reside en los retardos de la propia OUR; el error crece de ≈3.4 (H=6) a ≈4.4 (H=48)
y el R² cae de ≈0.35 a ≈0.15, reflejando la dificultad de la predicción recursiva
multi-paso a dos días vista. Estos valores sirven de referencia para los modelos
fundacionales de los cuadernos 04 y 05.